# Analytical Query Patterns

## Overview
Data warehouses are built for analytical queries: period comparisons, running totals, cohort analysis, and top-N ranking. This notebook covers the most common patterns using window functions on the insurance DWH schema.

**Window function quick reference:**
```sql
-- Year-over-year comparison:
LAG(metric, 1)  OVER (PARTITION BY product_line ORDER BY year)

-- Running total:
SUM(metric)     OVER (PARTITION BY product_line ORDER BY year ROWS UNBOUNDED PRECEDING)

-- 3-period moving average:
AVG(metric)     OVER (ORDER BY year ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING)

-- Top-N per group:
RANK()          OVER (PARTITION BY region ORDER BY total_premium DESC)
```

---

In [1]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


Data warehouse dataset loaded:
  dim_date              : 1096 rows
  dim_customer          : 6 rows
  dim_product           : 7 rows
  dim_agent             : 4 rows
  fact_policy           : 18 rows

  Total premiums: $45,600


---
## Year-over-year comparison

In [2]:
print("=== Period-over-period: year-over-year comparison ===")
print()

# YoY premium growth using window function LAG
rows = conn.execute("""
    WITH yearly AS (
        SELECT dd.year,
               dp.product_line,
               SUM(fp.premium_amount) AS premium
        FROM   fact_policy fp
        JOIN   dim_date    dd ON dd.date_key    = fp.date_key
        JOIN   dim_product dp ON dp.product_key = fp.product_key
        GROUP  BY dd.year, dp.product_line
    )
    SELECT year, product_line, premium,
           LAG(premium) OVER (PARTITION BY product_line ORDER BY year) AS prev_year,
           ROUND(100.0 * (premium - LAG(premium) OVER (PARTITION BY product_line ORDER BY year))
                       / NULLIF(LAG(premium) OVER (PARTITION BY product_line ORDER BY year), 0), 1) AS yoy_pct
    FROM yearly
    ORDER BY product_line, year
""").fetchall()

print("Year-over-year premium by product line:")
print(f"  {'year':<6s}  {'product_line':<10s}  {'premium':>10s}  {'prev_year':>10s}  {'YoY%':>7s}")
print("  " + "-"*50)
for r in rows:
    yoy = f"{r['yoy_pct']:+.1f}%" if r['yoy_pct'] is not None else "  (base)"
    prev = f"${r['prev_year']:,.0f}" if r['prev_year'] else "—"
    print(f"  {r['year']:<6d}  {r['product_line']:<10s}  ${r['premium']:>9,.0f}  {prev:>10s}  {yoy:>7s}")

print()
print("PostgreSQL YoY with window function:")
print("""
  WITH monthly AS (
      SELECT dd.year, dd.month,
             SUM(fp.premium_amount) AS premium
      FROM   fact_policy fp
      JOIN   dim_date dd ON dd.date_key = fp.date_key
      GROUP  BY dd.year, dd.month
  )
  SELECT year, month, premium,
         LAG(premium, 12) OVER (ORDER BY year, month) AS same_month_prev_year,
         ROUND(100.0 * (premium - LAG(premium,12) OVER (ORDER BY year, month))
               / NULLIF(LAG(premium,12) OVER (ORDER BY year, month), 0), 1) AS yoy_pct
  FROM   monthly
  ORDER  BY year, month;
  -- LAG(col, 12) = same month, 12 periods ago = year-over-year for monthly data
""")


=== Period-over-period: year-over-year comparison ===

Year-over-year premium by product line:
  year    product_line     premium   prev_year     YoY%
  --------------------------------------------------
  2022    auto        $    5,000           —    (base)
  2023    auto        $    2,600      $5,000   -48.0%
  2024    auto        $    4,900      $2,600   +88.5%
  2023    health      $    4,800           —    (base)
  2024    health      $    5,200      $4,800    +8.3%
  2022    home        $    3,800           —    (base)
  2023    home        $    9,100      $3,800  +139.5%
  2024    home        $    3,500      $9,100   -61.5%
  2022    life        $      600           —    (base)
  2023    life        $    5,400        $600  +800.0%
  2024    life        $      700      $5,400   -87.0%

PostgreSQL YoY with window function:

  WITH monthly AS (
      SELECT dd.year, dd.month,
             SUM(fp.premium_amount) AS premium
      FROM   fact_policy fp
      JOIN   dim_date dd ON dd.d

---
## Running totals and moving averages

In [3]:
print("=== Running totals and moving averages ===")
print()

rows = conn.execute("""
    WITH by_year_product AS (
        SELECT dd.year, dp.product_line,
               SUM(fp.premium_amount) AS premium,
               SUM(fp.claim_amount)   AS claims
        FROM   fact_policy fp
        JOIN   dim_date    dd ON dd.date_key    = fp.date_key
        JOIN   dim_product dp ON dp.product_key = fp.product_key
        GROUP  BY dd.year, dp.product_line
    )
    SELECT year, product_line, premium, claims,
           SUM(premium) OVER (PARTITION BY product_line ORDER BY year
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
               AS running_total_premium,
           ROUND(AVG(claims) OVER (PARTITION BY product_line ORDER BY year
                                   ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), 0)
               AS claims_3yr_moving_avg
    FROM   by_year_product
    ORDER  BY product_line, year
""").fetchall()

print("Running premium total and 3-year moving average claims:")
print(f"  {'product_line':<10s}  {'year':<6s}  {'premium':>10s}  {'running_total':>14s}  {'claims':>8s}  {'3yr_avg':>8s}")
print("  " + "-"*68)
for r in rows:
    avg = f"{r['claims_3yr_moving_avg']:,.0f}" if r['claims_3yr_moving_avg'] is not None else "—"
    print(f"  {r['product_line']:<10s}  {r['year']:<6d}  ${r['premium']:>9,.0f}  "
          f"${r['running_total_premium']:>13,.0f}  ${r['claims']:>7,.0f}  {avg:>8s}")


=== Running totals and moving averages ===

Running premium total and 3-year moving average claims:
  product_line  year       premium   running_total    claims   3yr_avg
  --------------------------------------------------------------------
  auto        2022    $    5,000  $        5,000  $  2,700     1,350
  auto        2023    $    2,600  $        7,600  $      0     1,167
  auto        2024    $    4,900  $       12,500  $    800       400
  health      2023    $    4,800  $        4,800  $      0         0
  health      2024    $    5,200  $       10,000  $      0         0
  home        2022    $    3,800  $        3,800  $    600     1,050
  home        2023    $    9,100  $       12,900  $  1,500       700
  home        2024    $    3,500  $       16,400  $      0       750
  life        2022    $      600  $          600  $      0         0
  life        2023    $    5,400  $        6,000  $      0         0
  life        2024    $      700  $        6,700  $      0         0

---
## Top-N per group ranking

In [4]:
print("=== Top-N per group: ranking agents within regions ===")
print()

rows = conn.execute("""
    WITH agent_summary AS (
        SELECT da.region,
               da.agent_name,
               da.channel,
               SUM(fp.premium_amount)  AS total_premium,
               COUNT(fp.policy_count)  AS n_policies,
               SUM(fp.claim_amount)    AS total_claims,
               RANK() OVER (PARTITION BY da.region ORDER BY SUM(fp.premium_amount) DESC)
                   AS rank_in_region
        FROM   fact_policy fp
        JOIN   dim_agent   da ON da.agent_key = fp.agent_key
        GROUP  BY da.region, da.agent_name, da.channel
    )
    SELECT region, rank_in_region, agent_name, channel,
           total_premium, n_policies, total_claims
    FROM   agent_summary
    WHERE  rank_in_region <= 2
    ORDER  BY region, rank_in_region
""").fetchall()

print("Top 2 agents per region by total premium:")
print(f"  {'region':<10s}  {'rank':<5s}  {'agent':<18s}  {'channel':<8s}  {'premium':>10s}  {'policies':>9s}")
print("  " + "-"*65)
for r in rows:
    print(f"  {r['region']:<10s}  #{r['rank_in_region']:<4d}  {r['agent_name']:<18s}  "
          f"{r['channel']:<8s}  ${r['total_premium']:>9,.0f}  {r['n_policies']:>9d}")

print()
print("RANK vs DENSE_RANK vs ROW_NUMBER:")
ranking_fns = [
    ("ROW_NUMBER()", "Always unique 1,2,3,4... even for ties",
     "Top-N where you need exactly N rows"),
    ("RANK()",       "Ties get same rank; next rank skips (1,1,3)",
     "True ranking where ties are acknowledged"),
    ("DENSE_RANK()", "Ties get same rank; no gaps (1,1,2,3)",
     "Percentile bands, tier assignment with ties"),
    ("NTILE(n)",     "Distributes rows into n equal buckets",
     "Quartile/decile analysis"),
    ("PERCENT_RANK()","Relative rank as 0.0–1.0 percentile",
     "Where does this agent rank in the distribution?"),
]
for fn, behaviour, use in ranking_fns:
    print(f"  {fn:<18s}  {behaviour:<42s}  {use}")


=== Top-N per group: ranking agents within regions ===

Top 2 agents per region by total premium:
  region      rank   agent               channel      premium   policies
  -----------------------------------------------------------------
  Central     #1     Priya Sharma        online    $   11,300          4
  East        #1     Sandra Lee          direct    $   13,300          6
  East        #2     Marc Tremblay       broker    $    9,000          3
  West        #1     Tom Okafor          broker    $   12,000          5

RANK vs DENSE_RANK vs ROW_NUMBER:
  ROW_NUMBER()        Always unique 1,2,3,4... even for ties      Top-N where you need exactly N rows
  RANK()              Ties get same rank; next rank skips (1,1,3)  True ranking where ties are acknowledged
  DENSE_RANK()        Ties get same rank; no gaps (1,1,2,3)       Percentile bands, tier assignment with ties
  NTILE(n)            Distributes rows into n equal buckets       Quartile/decile analysis
  PERCENT_RANK()      R

---
## Common Pitfalls

**1. LAG/LEAD returning NULL for first/last row — breaking YoY calculation**
`LAG(premium) OVER (ORDER BY year)` returns NULL for the first year. `100 * (premium - NULL) / NULL` returns NULL. Use `NULLIF` and `COALESCE` explicitly: `ROUND(100.0 * (premium - LAG(premium)) / NULLIF(LAG(premium), 0), 1)` and document that the base year has no YoY value.

**2. Using RANK() when you need exactly N rows per group**
`RANK() <= 2` with ties can return 3+ rows per group (tied agents both get rank=1, then no rank=2). Use `ROW_NUMBER() <= 2` if you need exactly 2 rows per group, and specify a deterministic tiebreaker in ORDER BY.

**3. Window function evaluated before WHERE — no filter on window result**
`SELECT *, RANK() OVER (...) AS r FROM fact_policy WHERE r = 1` fails — window functions are evaluated after WHERE. Use a CTE or subquery: `WITH ranked AS (...) SELECT * FROM ranked WHERE rank_col = 1`.

**4. Running total restarting unexpectedly**
`SUM(premium) OVER (ORDER BY year)` accumulates across all partitions if `PARTITION BY` is omitted. Add `PARTITION BY product_line` to get a running total that resets for each product line.

**5. Period comparison breaking across year boundaries**
`LAG(premium, 12)` for monthly data works only if there are no gaps. A missing month shifts all subsequent LAG values by one. Join to dim_date to fill in zero-premium months before applying LAG.

**6. Loss ratio averaging instead of computing**
`AVG(loss_ratio_pct)` averages pre-computed ratios, which is mathematically wrong for groups of different sizes. Always recompute: `SUM(claims)/SUM(premium)` on the aggregated data, never `AVG` of stored ratio values.


---
*sql_methods_library - Samantha McGarrigle*